# VeriqURL Full Pipeline Notebook

**Project:** VeriqURL — Lightweight URL-Based Phishing Detection and Awareness Tool  
**Purpose:** This notebook explains the full project pipeline from dataset preparation to model evaluation, threshold validation, explanation rules, and case study prediction.

This notebook is intended for project demonstration and supervisor review. It does not replace the production Streamlit app (`app.py`).

## 1. Project Setup

**Purpose:** Configure project paths and import the main Python libraries used throughout the project.

In [19]:
from pathlib import Path
import sys
import os
import json
import pandas as pd
import numpy as np

# Try to detect project root automatically
current_dir = Path.cwd()

if (current_dir / "src").exists() and (current_dir / "app.py").exists():
    ROOT_DIR = current_dir
elif (current_dir.parent / "src").exists() and (current_dir.parent / "app.py").exists():
    ROOT_DIR = current_dir.parent
else:
    raise FileNotFoundError(
        "Project root not found. Please open this notebook from the main "
        "'Phishing Detection' project folder."
    )

SRC_DIR = ROOT_DIR / "src"
RESULTS_DIR = ROOT_DIR / "results"
MODELS_DIR = ROOT_DIR / "models"
DATA_DIR = ROOT_DIR / "data"

if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

print("Current working directory:", current_dir)
print("Detected project root:", ROOT_DIR)
print("src exists:", SRC_DIR.exists())
print("results exists:", RESULTS_DIR.exists())
print("models exists:", MODELS_DIR.exists())
print("data exists:", DATA_DIR.exists())

Current working directory: c:\Users\Wen Tao\Desktop\Studies\FYP\Phishing Detection\notebooks
Detected project root: c:\Users\Wen Tao\Desktop\Studies\FYP\Phishing Detection
src exists: True
results exists: True
models exists: True
data exists: True


## 2. Dataset Loading and Cleaning

**Purpose:** Show where the dataset is stored and how the cleaned dataset is loaded. The final system uses URL-only detection, so only URL text and labels are needed.

The cleaning script used in the project is `src/preprocess.py`.

In [20]:
cleaned_path = DATA_DIR / "processed" / "kaggle_cleaned.csv"

if cleaned_path.exists():
    df = pd.read_csv(cleaned_path)
    print("Cleaned dataset shape:", df.shape)
    display(df.head())
else:
    print("Cleaned dataset not found. Run: python src/preprocess.py")

Cleaned dataset shape: (235370, 2)


,url,label
0,https://www.southbankmosaics.com,0
1,https://www.uni-mainz.de,0
2,https://www.voicefmradio.co.uk,0
3,https://www.sfnmjournal.com,0
4,https://www.rewildingargentina.org,0


## 3. Train / Validation / Test Split

**Purpose:** Load the generated train, validation, and test splits. These splits are used consistently for Random Forest and TCN evaluation.

The split script used in the project is `src/split_data.py`.

In [21]:
split_dir = DATA_DIR / "processed" / "splits"
for name in ["train", "val", "test"]:
    path = split_dir / f"{name}.csv"
    if path.exists():
        split_df = pd.read_csv(path)
        print(f"{name}: {split_df.shape}")
        if "label" in split_df.columns:
            print(split_df["label"].value_counts().to_dict())
    else:
        print(f"Missing: {path}")

train: (164759, 2)
{0: 94395, 1: 70364}
val: (35305, 2)
{0: 20227, 1: 15078}
test: (35306, 2)
{0: 20228, 1: 15078}


## 4. Random Forest Feature Extraction

**Purpose:** Demonstrate the handcrafted lexical and structural URL features used by the Random Forest model.

The feature extraction script used in the project is `src/feature_extraction.py`. The deployed app uses RF-compatible features through `extract_model_features()` in `src/explanation_rules.py` to ensure the prediction input matches the trained model.

In [22]:
from explanation_rules import extract_model_features, extract_explanation_indicators

sample_url = "http://secure-paypal-login-verification.com/account/verify"
features = extract_model_features(sample_url)
print("RF-compatible model features:")
display(pd.DataFrame([features]).T.rename(columns={0: "value"}))

RF-compatible model features:


,value
url_length,58
domain_length,36
path_length,15
query_length,0
digit_count,0
special_char_count,9
dot_count,1
hyphen_count,3
slash_count,4
question_mark_count,0


## 5. Random Forest Training

**Purpose:** Explain how the Random Forest baseline was trained. Random Forest is the lightweight feature-based baseline model.

The full training script is `src/train_rf.py`. If the model already exists, this notebook does not need to retrain it.

In [23]:
rf_model_path = MODELS_DIR / "random_forest_model.pkl"
print("RF model exists:", rf_model_path.exists())
print("RF model path:", rf_model_path)

# To retrain manually, run this in terminal:
# python src/train_rf.py

RF model exists: True
RF model path: c:\Users\Wen Tao\Desktop\Studies\FYP\Phishing Detection\models\random_forest_model.pkl


## 6. TCN Character Vocabulary and URL Encoding

**Purpose:** Show how URLs are transformed into fixed-length character-level sequences for the TCN model.

The TCN data preparation script is `src/prepare_tcn_data.py`.

In [24]:
vocab_path = DATA_DIR / "processed" / "tcn" / "char_vocab.json"

if vocab_path.exists():
    with open(vocab_path, "r", encoding="utf-8") as f:
        char_to_idx = json.load(f)
    print("Vocabulary size:", len(char_to_idx))
    print("First 20 vocabulary items:", list(char_to_idx.items())[:20])
else:
    print("char_vocab.json not found. Run: python src/prepare_tcn_data.py")

Vocabulary size: 71
First 20 vocabulary items: [('!', 1), ('#', 2), ('$', 3), ('%', 4), ('&', 5), ("'", 6), ('(', 7), (')', 8), ('*', 9), ('+', 10), (',', 11), ('-', 12), ('.', 13), ('/', 14), ('0', 15), ('1', 16), ('2', 17), ('3', 18), ('4', 19), ('5', 20)]


In [25]:
from predict_compare import encode_url_for_tcn

if vocab_path.exists():
    encoded = encode_url_for_tcn(sample_url, char_to_idx, max_len=200)
    print("Encoded sequence length:", len(encoded))
    print("Non-padding character count:", int(np.count_nonzero(encoded)))
    print("First 60 encoded values:", encoded[:60])

Encoded sequence length: 200
Non-padding character count: 58
First 60 encoded values: [40 52 52 48 25 14 14 51 37 35 53 50 37 12 48 33 57 48 33 44 12 44 47 39
 41 46 12 54 37 50 41 38 41 35 33 52 41 47 46 13 35 47 45 14 33 35 35 47
 53 46 52 14 54 37 50 41 38 57  0  0]


## 7. TCN Model Architecture

**Purpose:** Show the TCN architecture used in the final implementation. The model uses character embedding, temporal convolutional blocks, global max pooling, and a fully connected output layer.

The training script is `src/train_tcn.py`. Full TCN retraining is not run inside this notebook because it can take a long time.

In [26]:
from predict_compare import TCNClassifier

if vocab_path.exists():
    model = TCNClassifier(
        vocab_size=len(char_to_idx),
        embed_dim=32,
        num_channels=64,
        kernel_size=3,
        dropout=0.2,
    )
    print(model)
    total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print("Trainable parameters:", total_params)

TCNClassifier(
  (embedding): Embedding(72, 32, padding_idx=0)
  (tcn): Sequential(
    (0): TemporalBlock(
      (net): Sequential(
        (0): Conv1d(32, 64, kernel_size=(3,), stride=(1,), padding=(2,))
        (1): Chomp1d()
        (2): ReLU()
        (3): Dropout(p=0.2, inplace=False)
        (4): Conv1d(64, 64, kernel_size=(3,), stride=(1,), padding=(2,))
        (5): Chomp1d()
        (6): ReLU()
        (7): Dropout(p=0.2, inplace=False)
      )
      (downsample): Conv1d(32, 64, kernel_size=(1,), stride=(1,))
      (relu): ReLU()
    )
    (1): TemporalBlock(
      (net): Sequential(
        (0): Conv1d(64, 64, kernel_size=(3,), stride=(1,), padding=(4,), dilation=(2,))
        (1): Chomp1d()
        (2): ReLU()
        (3): Dropout(p=0.2, inplace=False)
        (4): Conv1d(64, 64, kernel_size=(3,), stride=(1,), padding=(4,), dilation=(2,))
        (5): Chomp1d()
        (6): ReLU()
        (7): Dropout(p=0.2, inplace=False)
      )
      (relu): ReLU()
    )
    (2): Tempora

## 8. TCN Training Summary

**Purpose:** Display saved TCN training history and summary. This provides evidence of the training process without retraining the model in the notebook.

In [27]:
history_path = RESULTS_DIR / "tcn_training_history.csv"
summary_path = RESULTS_DIR / "tcn_training_summary.csv"

if history_path.exists():
    history_df = pd.read_csv(history_path)
    print("TCN training history:")
    display(history_df.tail())
else:
    print("tcn_training_history.csv not found.")

if summary_path.exists():
    summary_df = pd.read_csv(summary_path)
    print("TCN training summary:")
    display(summary_df)
else:
    print("tcn_training_summary.csv not found.")

TCN training history:


,epoch,train_loss,val_accuracy,val_precision,val_recall,val_f1
5,6,0.013983,0.998017,1.000000,0.995357,0.997673
6,7,0.013302,0.997989,0.999800,0.995490,0.997640
7,8,0.013023,0.997961,0.999867,0.995357,0.997607
8,9,0.012727,0.997961,0.999734,0.995490,0.997607
9,10,0.012078,0.997961,0.999933,0.995291,0.997607


TCN training summary:


,model,training_time_s,epochs,batch_size,learning_rate,embed_dim,num_channels,kernel_size,dropout,best_val_f1
0,TCN,6433.383138,10,128,0.0005,32,64,3,0.2,0.997673


## 9. Model Evaluation

**Purpose:** Load the saved model evaluation results for Random Forest and TCN. These metrics are reported in the final report.

In [28]:
for filename in ["rf_metrics.csv", "tcn_metrics.csv", "tcn_confusion_matrix.csv"]:
    path = RESULTS_DIR / filename
    print("Displaying:", filename)
    if path.exists():
        display(pd.read_csv(path))
    else:
        print("Missing:", path)

Displaying: rf_metrics.csv


,dataset,accuracy,precision,recall,f1,inference_time,per_sample_time,training_time
0,Validation,0.995185,0.999130,0.989587,0.994336,0.085464,0.000002,3.462468
1,Test,0.996006,0.998665,0.991975,0.995309,0.084455,0.000002,3.462468


Displaying: tcn_metrics.csv


,model,accuracy,precision,recall,f1_score,pred_positive_rate,training_time_s,testing_time_s,per_sample_time_s,per_sample_latency_ms,tes,ies,rtde,model_size_mb,num_parameters
0,TCN,0.998527,0.999933,0.996618,0.998273,0.42565,6433.383138,23.939396,0.000678,0.678055,0.000155,0.041711,1.472635,0.282927,72449


Displaying: tcn_confusion_matrix.csv


,Unnamed: 0,predicted_benign,predicted_phishing
0,actual_benign,20227,1
1,actual_phishing,51,15027


## 10. Model Comparison and Lightweight Metrics

**Purpose:** Compare RF and TCN using both classification metrics and lightweight efficiency metrics such as TES, IES, and RTDE.

In [29]:
comparison_path = RESULTS_DIR / "model_comparison.csv"
if comparison_path.exists():
    comparison_df = pd.read_csv(comparison_path)
    display(comparison_df)
else:
    print("model_comparison.csv not found. Run: python src/compare_models.py")

,model,accuracy,precision,recall,f1_score,training_time_s,testing_time_s,per_sample_time_s,per_sample_latency_ms,tes,ies,rtde,model_size_mb,num_parameters,pred_positive_rate
0,Random Forest,0.996006,0.998665,0.991975,0.995309,3.462468,0.084455,0.000002,0.002392,0.287658,11.793303,416.374344,NaN,NaN,NaN
1,TCN,0.998527,0.999933,0.996618,0.998273,6433.383138,23.939396,0.000678,0.678055,0.000155,0.041711,1.472635,0.282927,72449.0,0.42565


## 11. Risk Threshold Validation

**Purpose:** Explain why the High Risk threshold is set to 0.80. Candidate thresholds were tested on the validation set to compare High Risk precision, recall, false positives, and Suspicious category size.

The validation script is `src/validate_risk_thresholds.py`.

In [30]:
threshold_path = RESULTS_DIR / "risk_threshold_validation.csv"
distribution_path = RESULTS_DIR / "risk_threshold_distribution.csv"

if threshold_path.exists():
    threshold_df = pd.read_csv(threshold_path)
    display(threshold_df)
else:
    print("risk_threshold_validation.csv not found. Run: python src/validate_risk_thresholds.py")

if distribution_path.exists():
    print("Risk distribution by threshold:")
    display(pd.read_csv(distribution_path))

,candidate_high_risk_threshold,base_binary_accuracy_at_0_50,base_binary_precision_at_0_50,base_binary_recall_at_0_50,base_binary_f1_at_0_50,high_risk_precision,high_risk_recall,high_risk_count,high_risk_true_phishing,high_risk_false_positive,suspicious_count,suspicious_true_phishing,suspicious_benign,low_risk_count,low_risk_phishing,low_risk_benign,total_samples,total_phishing,total_benign
0,0.6,0.997904,0.999933,0.995159,0.99754,1.0,0.990848,14940,14940,0,66,65,1,20299,73,20226,35305,15078,20227
1,0.7,0.997904,0.999933,0.995159,0.99754,1.0,0.990052,14928,14928,0,78,77,1,20299,73,20226,35305,15078,20227
2,0.8,0.997904,0.999933,0.995159,0.99754,1.0,0.989256,14916,14916,0,90,89,1,20299,73,20226,35305,15078,20227
3,0.9,0.997904,0.999933,0.995159,0.99754,1.0,0.987664,14892,14892,0,114,113,1,20299,73,20226,35305,15078,20227


Risk distribution by threshold:


,candidate_high_risk_threshold,low_risk_count,suspicious_count,high_risk_count,low_risk_phishing,suspicious_true_phishing,high_risk_true_phishing,high_risk_false_positive
0,0.6,20299,66,14940,73,65,14940,0
1,0.7,20299,78,14928,73,77,14928,0
2,0.8,20299,90,14916,73,89,14916,0
3,0.9,20299,114,14892,73,113,14892,0


## 12. Evidence-based Explanation Layer

**Purpose:** Demonstrate how VeriqURL extracts explanation indicators from a URL. These indicators are used to generate human-readable reasons and evidence-based awareness guidance.

In [31]:
indicators = extract_explanation_indicators(sample_url)
print("Explanation indicators for:", sample_url)
display(pd.DataFrame([indicators]).T.rename(columns={0: "value"}))

Explanation indicators for: http://secure-paypal-login-verification.com/account/verify


,value
scheme,http
domain,secure-paypal-login-verification.com
registered_domain,secure-paypal-login-verification.com
path,/account/verify
query,
url_length,58
domain_length,36
path_length,15
query_length,0
digit_count,0


## 13. Case Study Prediction

**Purpose:** Run the final prediction pipeline on representative benign, phishing, and suspicious URLs. This demonstrates the same logic used by the Streamlit app.

In [32]:
import torch
from predict_compare import (
    load_rf_model,
    load_char_vocab,
    load_tcn_model,
    predict_rf,
    predict_tcn,
    get_final_decision,
)
from explanation_rules import build_explanation_result

# Ensure relative paths inside predict_compare.py work correctly
os.chdir(ROOT_DIR)

case_urls = [
    "https://www.google.com",
    "http://secure-paypal-login-verification.com/account/verify",
    "https://update-service-login.example.org",
]

if rf_model_path.exists() and (MODELS_DIR / "tcn_model.pt").exists() and vocab_path.exists():
    rf_model = load_rf_model()
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    char_to_idx = load_char_vocab()
    tcn_model = load_tcn_model(len(char_to_idx), device)

    rows = []
    for url in case_urls:
        rf_result = predict_rf(url, rf_model)
        tcn_result = predict_tcn(url, tcn_model, char_to_idx, device)
        final_decision = get_final_decision(rf_result, tcn_result)
        explanation = build_explanation_result(
            url,
            max(rf_result["phishing_probability"], tcn_result["phishing_probability"])
        )
        rows.append({
            "url": url,
            "rf_label": rf_result["predicted_label"],
            "rf_phishing_probability": round(rf_result["phishing_probability"], 4),
            "rf_risk": rf_result["risk_level"],
            "tcn_label": tcn_result["predicted_label"],
            "tcn_phishing_probability": round(tcn_result["phishing_probability"], 4),
            "tcn_risk": tcn_result["risk_level"],
            "average_phishing_probability": round(final_decision["average_phishing_probability"], 4),
            "final_label": final_decision["final_label"],
            "final_risk": final_decision["final_risk"],
            "top_reason": explanation["reasons"][0],
            "top_awareness_guidance": explanation["awareness_guidance"][0],
        })
    display(pd.DataFrame(rows))
else:
    print("Model files are missing. Case study prediction requires saved model artifacts.")

,url,rf_label,rf_phishing_probability,rf_risk,tcn_label,tcn_phishing_probability,tcn_risk,average_phishing_probability,final_label,final_risk,top_reason,top_awareness_guidance
0,https://www.google.com,benign,0.0022,Low Risk,benign,0.0017,Low Risk,0.0020,benign,Low Risk,No strong suspicious lexical pattern was detec...,No strong suspicious lexical pattern was detec...
1,http://secure-paypal-login-verification.com/ac...,phishing,1.0000,High Risk,phishing,1.0000,High Risk,1.0000,phishing,High Risk,The URL uses HTTP instead of HTTPS.,"Do not open the link or enter login, banking, ..."
2,https://update-service-login.example.org,phishing,0.7384,Suspicious,phishing,1.0000,High Risk,0.8692,phishing,High Risk,The URL contains suspicious or suggestive word...,"Do not open the link or enter login, banking, ..."


## 14. Summary

**Purpose:** Summarise what the pipeline demonstrates.

This notebook shows the complete VeriqURL workflow: dataset preparation, RF feature extraction, TCN sequence encoding, model training artefacts, evaluation metrics, lightweight comparison, threshold validation, explanation indicators, and case study prediction. The deployed Streamlit app uses the saved model artefacts and explanation rules to provide URL analysis and awareness guidance to users.